## Полный гайд по dunder-методам в Python (от новичка до профи)

https://habr.com/ru/articles/1033432/

### Object life-cycle

__new__(cls): Настоящий конструктор

Именно __new__ отвечает за создание объекта. Он выделяет память и возвращает ту самую пустую «болванку», которая затем передается в __init__. Обратите внимание: метод принимает класс (cls), а не экземпляр (self), потому что экземпляра на этот момент физически не существует.

В 99% случаев вам не нужно трогать __new__. Базовый класс object отлично справляется сам. Но если вам нужно вмешаться в процесс создания — например, реализовать паттерн Singleton (когда в системе может существовать только один экземпляр класса), — без __new__ не обойтись.

In [23]:
class DatabaseConnection:
    _instance = None

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            # If object doesn't exist, create it through super class
            cls._instance = super().__new__(cls)
        
        # If exists, returns existed
        return cls._instance
    
# Validation
db1 = DatabaseConnection()
db2 = DatabaseConnection()
print(db1 is db2)

True


__init__(self): Инициализатор

Итак, __new__ построил «коробку» и вернул её. Теперь за дело берется __init__. Его единственная задача — наполнить эту коробку данными (задать начальное состояние атрибутов).

In [2]:
class User:
    def __init__(self, username: str):
        # Object (self) is already created, need to fill up attributes
        self.username = username


__del__(self): Разрушитель (и почему ему нельзя доверять)

Метод __del__ вызывается сборщиком мусора (Garbage Collector), когда счетчик ссылок на объект падает до нуля

In [ ]:
class DangerFileWrapper:
    def __init__(self, filename: str):
        self.file = open(filename, 'w')

    def __del__(self):
        # ПЛОХАЯ ПРАКТИКА! Файл может остаться открытым или 
        # упасть с ошибкой при завершении скрипта.
        self.file.close() 

### How we see the object

__str__(self): Для людей
Этот метод должен возвращать красивое, человекочитаемое описание объекта. Его цель — быть понятным конечному пользователю.

Он вызывается автоматически, когда вы используете print(obj), явно приводите объект к строке str(obj) или вставляете его в f-строку f"Мой объект: {obj}".

__repr__(self): Для разработчиков
__repr__ (от representation) — это технический паспорт объекта. Его задача — показать однозначную информацию для отладки и логирования.

In [8]:
class User:
    def __init__(self, username: str, role: str):
        self.username = username
        self.role = role

    def __str__(self):
        # Beautified output for UI/user
        return f"User: {self.username}, (Role: {self.role})"
    
    def __repr__(self):
        # Unique output for logs/developer console
        return f"User('{self.username}', '{self.role}')"
    
user = User("alice", "admin")

# 1. How is user is seen
print(user)
print(f"Hello, {user}")

# 2. How is interpreator/develop is seen
print(repr(user))
print([user])

# 3. Check for golden rule
user_clone = eval(repr(user))
print(user_clone.username)

User: alice, (Role: admin)
Hello, User: alice, (Role: admin)
User('alice', 'admin')
[User('alice', 'admin')]
alice


### Operator overloading. Arithmetic and comparision

Главное правило обычной арифметики: она не должна менять исходные объекты.
Результатом сложения двух объектов должен быть новый объект.

In [13]:
# Math: __add__ (+), __sub__ (-), __mul__ (*)
class Vector:
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y
    
    def __add__(self, other):
        if not isinstance(other, Vector):
            # return NotImplemented, not throwing exception
            # Gives chance to try other adding methods
            return NotImplemented
        
        return Vector(self.x + other.x, self.y + other.y)
    
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"
    
v1 = Vector(1, 2)
v2 = Vector(3, 4)

print(v1 + v2)

Vector(4, 6)


Отраженная (Right) арифметика: __radd__

А что если мы захотим прибавить к нашему вектору просто число (скаляр), сдвинув его координаты? Допишем проверку в __add__. Но что произойдет при 2 + v1?

In [15]:
#  Math: __radd__
class Vector:
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y
    
    def __add__(self, other):
        if isinstance(other, Vector):
            return Vector(self.x + other.x, self.y + other.y)
        
        if isinstance(other, int):
            return Vector(self.x + other, self.y + other)
        
        return NotImplemented
    
    def __radd__(self, other):
        # The same logic, just call __add__
        return self.__add__(other)
    
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"
    
v1 = Vector(1, 2)
print(v1 + 10)  # Сработает __add__ -> Vector(11, 12)
print(10 + v1)  # Сработает __radd__ -> Vector(11, 12)

Vector(11, 12)
Vector(11, 12)


In-place арифметика: __iadd__ (+=)
Оператор += модифицирует объект на месте (in-place).

Если вы не напишете __iadd__, Python просто сделает v1 = v1 + v2 (создаст новый объект и перезапишет переменную). Но если ваш объект тяжелый (например, большая матрица данных), постоянное создание копий убьет производительность. __iadd__ позволяет изменить текущий объект и **обязательно должен вернуть self**.

In [16]:
# In place arithmetic: __iadd__ (+=)
class Vector:
    def __init__(self, x: int, y: int):
        self.x = x
        self.y = y
    
    def __add__(self, other):
        if isinstance(other, Vector):
            return Vector(self.x + other.x, self.y + other.y)
        
        if isinstance(other, int):
            return Vector(self.x + other, self.y + other)
        
        return NotImplemented
    
    def __iadd__(self, other):
        if isinstance(other, Vector):
            self.x = other.x
            self.y = other.y
            return self
        
        return NotImplemented
    
    def __radd__(self, other):
        # The same logic, just call __add__
        return self.__add__(other)
    
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"


v1 = Vector(1, 1)
print(id(v1)) # Запоминаем адрес в памяти
v1 += Vector(2, 2)
print(id(v1)) # Адрес тот же! Мы изменили сам объект, а не создали копию.

4442664176
4442664176


Сравнения: __eq__ (==), __lt__ (<), __gt__ (>)
По умолчанию объекты пользовательских классов равны (==), только если это один и тот же объект в памяти. Это редко бывает полезно.

Для полноценного сравнения нужно реализовать 6 методов: __eq__ (==), __ne__ (!=), __lt__ (<), __le__ (<=), __gt__ (>), __ge__ (>=). Писать их все — невероятно скучно и нарушает принцип DRY.

Тут на сцену выходит декоратор @total_ordering из встроенного модуля functools. Лайфхак для ленивых профи: достаточно реализовать метод __eq__ и один любой метод неравенства (например, __lt__). Всю остальную магию декоратор сгенерирует за вас.

In [17]:
# Comparision: __eq__ (==), __lt__ (<), __gt__ (>)

from functools import total_ordering

@total_ordering
class Player:
    def __init__(self, name: str, score: int):
        self.name = name
        self.score = score

    # Проверка на равенство
    def __eq__(self, other):
        if not isinstance(other, Player):
            return NotImplemented
        return self.score == other.score

    # Проверка "Меньше" (Less Than)
    def __lt__(self, other):
        if not isinstance(other, Player):
            return NotImplemented
        return self.score < other.score

p1 = Player("NoobMaster", 100)
p2 = Player("ProGamer", 9000)

print(p1 == p2)  # False (из __eq__)
print(p1 < p2)   # True (из __lt__)
print(p1 >= p2)  # False (сгенерировано total_ordering!)

False
True
False


### Длина: __len__ и сишная оптимизация

In [18]:
class Playlist:
    def __init__(self, name: str, tracks: list):
        self.name = name
        self.tracks = tracks

    def __len__(self):
        # Возвращаем количество треков
        return len(self.tracks)

my_music = Playlist("Road Trip", ["Track 1", "Track 2", "Track 3"])
print(len(my_music))  # Вывод: 3

3


### Индексация и срезы: __getitem__, __setitem__, __delitem__

Чтобы ваш объект начал понимать квадратные скобки, ему нужна эта троица.

__getitem__(self, key): для чтения val = obj[key]

__setitem__(self, key, value): для записи obj[key] = value

__delitem__(self, key): для удаления del obj[key]

Важный нюанс со срезами: Когда вы запрашиваете срез my_obj[1:5], Python не вызывает метод несколько раз. Он вызывает __getitem__ ровно один раз, но вместо индекса передает туда специальный объект slice. Ваш код должен уметь с ним работать.

In [19]:
class CustomList:
    def __init__(self, *args):
        self._data = list(args)

    def __getitem__(self, index):
        # index может быть числом (int) или срезом (slice)
        if isinstance(index, slice):
            print(f"Запрошен срез: старт={index.start}, стоп={index.stop}, шаг={index.step}")
            return self._data[index]
        return self._data[index]

    def __setitem__(self, index, value):
        self._data[index] = value

custom = CustomList(10, 20, 30, 40, 50)
print(custom[1])     # Вывод: 20
print(custom[1:4:2]) # Запрошен срез: старт=1, стоп=4, шаг=2. Вывод: [20, 40]


20
Запрошен срез: старт=1, стоп=4, шаг=2
[20, 40]


### Проверка вхождения: __contains__

Этот метод отвечает за то, как ваш объект реагирует на оператор in (например, if item in my_obj:)

In [20]:
class AccessControl:
    def __init__(self, allowed_users: list):
        # Конвертируем список в set для мгновенного поиска за O(1)
        self._allowed = set(allowed_users)

    def __contains__(self, user: str):
        # Вызывается при 'user in access_control'
        return user in self._allowed

door = AccessControl(["admin", "root", "ceo"])

print("admin" in door)  # True (сработает быстро через set)
print("hacker" in door) # False

True
False


## Управление доступом к атрибутам

__getattr__ vs __getattribute__: Классика собеседований
Если вас спросят на собеседовании о разнице между этими двумя методами, запомните главное правило: один — это тотальный диктатор, а другой — служба спасения.

__getattribute__(self, name) (Диктатор): Вызывается всегда и абсолютно при любом обращении к атрибуту. Неважно, существует атрибут или нет. Если вы переопределили этот метод, Python делегирует ему все полномочия. Использовать его нужно крайне редко, потому что он ломает стандартный механизм поиска атрибутов и работает медленно.

__getattr__(self, name) (Служба спасения): Вызывается только тогда, когда Python уже поискал атрибут в объекте, поискал в классе, поискал в родительских классах, ничего не нашел и готов выбросить AttributeError. Это идеальное место для реализации динамических API (например, когда вы хотите превратить ключи словаря в атрибуты).

Опасная ловушка: Бесконечная рекурсия в __getattribute__. Если внутри __getattribute__ вы напишете self.name или даже self.__dict__, Python снова вызовет __getattribute__. Произойдет рекурсия, и программа упадет с RecursionError. Чтобы получить реальное значение, всегда нужно использовать метод базового класса: super().__getattribute__(name).

In [25]:
class DynamicConfig:
    def __init__(self, data: dict):
        self._data = data

    def __getattr__(self, item):
        # Этот код сработает ТОЛЬКО если атрибута нет в классе
        if item in self._data:
            return self._data[item]
        # Если ключа нет и в словаре, честно бросаем ошибку
        raise AttributeError(f"В конфиге нет настройки '{item}'")

config = DynamicConfig({"host": "localhost", "port": 8080})

# 'port' нет как реального атрибута класса, но __getattr__ его перехватит:
print(config.port)  # Вывод: 8080
# print(config.password) # Бросит AttributeError

8080


__setattr__ и __delattr__: Перехват записи и удаления

Метод __setattr__(self, name, value) срабатывает каждый раз, когда вы пишете obj.name = value. Метод __delattr__(self, name) — при вызове del obj.name.

Как писать правильно? Есть два пути: либо обращаться напрямую к словарю атрибутов self.__dict__[name] = value (работает быстрее, но обходит логику родительских классов), либо использовать super().__setattr__(name, value) (безопаснее и предпочтительнее).


In [22]:
class ReadOnlyBox:
    def __init__(self, name: str):
        # Внутри __init__ тоже вызывается __setattr__!
        self.name = name

    def __setattr__(self, name, value):
        # Проверяем, есть ли уже такой атрибут
        if hasattr(self, name):
            raise TypeError(f"Атрибут '{name}' доступен только для чтения!")
        
        # Если атрибута еще нет (инициализация), используем super()
        super().__setattr__(name, value)

box = ReadOnlyBox("Секретная посылка")
print(box.name) # Вывод: Секретная посылка

# Попытка изменить существующий атрибут:
# box.name = "Пустая коробка" 
# Выбросит TypeError: Атрибут 'name' доступен только для чтения!

# А вот добавить новый — можно (если мы не запретили это отдельно)
box.weight = 10 

Секретная посылка


## Итераторы и Генераторы

Разница между Iterable и Iterator
Iterable (Итерируемый объект) — это коллекция данных. У него есть только один метод: __iter__. Этот метод обязан вернуть итератор.

Аналогия: Это книга. Вы можете её прочитать.

Iterator (Итератор) — это объект, который помнит текущее состояние (на каком элементе мы остановились) и умеет выдавать следующий элемент. У него есть метод __next__ (выдает значение) и метод __iter__ (возвращает самого себя — self).

Стандартный список (list) — это Iterable, но не Iterator. Вы не можете вызвать next(my_list). Но когда вы пишете for x in my_list, цикл под капотом незаметно вызывает iter(my_list), получает итератор и уже у нее запрашивает значения.

In [26]:
class Countdown:
    """Итератор для обратного отсчета."""
    def __init__(self, start: int):
        self.current = start

    def __iter__(self):
        # Итератор обязан возвращать самого себя
        return self

    def __next__(self):
        # Если дошли до нуля — бросаем специальное исключение
        if self.current <= 0:
            raise StopIteration
        
        # Запоминаем текущее значение, уменьшаем счетчик и отдаем
        result = self.current
        self.current -= 1
        return result
    
timer = Countdown(3)
for number in timer:
    print(number)

# Выведет: 3, 2, 1

3
2
1


### Эволюция: Генераторы (Функции с yield)

In [27]:
def countdown_gen(start: int):
    while start > 0:
        yield start
        start -= 1

for number in countdown_gen(3):
    print(number)

3
2
1


## Контекстные менеджеры (Оператор with)

### __enter__(self): Подготовка

Этот метод вызывается в самом начале блока with. Здесь вы открываете соединения, блокируете потоки (мьютексы) или инициализируете ресурсы.

Важно: То, что вернет __enter__, будет записано в переменную после ключевого слова as. Если вы пишете with open('file.txt') as f:, то f — это именно то, что вернул __enter__. Часто метод возвращает просто self.

### __exit__(self, exc_type, exc_val, traceback): Уборка и обработка ошибок

Этот метод гарантированно вызывается при выходе из блока with. Даже если внутри блока произошел return, break или скрипт упал с критической ошибкой.

Обратите внимание на три аргумента. Если внутри блока with всё прошло гладко, в них прилетит (None, None, None). Но если случился сбой, __exit__ получит полную информацию об ошибке: тип исключения, само значение и объект трейсбека (историю вызовов).

Этот метод — бронежилет вашего кода. Он гарантированно выполнится при выходе из блока with, **даже если внутри блока произошел сбой, исключение или был вызван return / break**.

Метод принимает три аргумента:

exc_type — класс исключения (например, ValueError).

exc_val — само значение исключения (сообщение об ошибке).

exc_tb — traceback (объект с историей вызовов).

Если внутри блока with всё прошло гладко, все три аргумента будут равны None.

In [28]:
class DBTransaction:
    def __init__(self, db_connection):
        self.db = db_connection

    def __enter__(self):
        print(">> Начинаем транзакцию...")
        # Возвращаем объект базы, чтобы с ним можно было работать внутри with
        return self.db 

    def __exit__(self, exc_type, exc_val, traceback):
        if exc_type is not None:
            # Если случилась ошибка (любая)
            print(f"<< Ошибка {exc_type.__name__}: {exc_val}. Делаем ROLLBACK!")
            # Делаем откат изменений
            # self.db.rollback()
            
            # Если мы хотим «проглотить» ошибку и не дать скрипту упасть:
            return True 
            
            # Если хотим, чтобы программа упала с этой ошибкой, 
            # возвращаем False или просто ничего не пишем.
        else:
            # Ошибок нет
            print("<< Всё отлично, делаем COMMIT!")
            # self.db.commit()


# Эмулируем базу данных
mock_db = {"user": "admin"}

print("--- Успешный сценарий ---")
with DBTransaction(mock_db) as db:
    db["score"] = 100
    print("Работаем с БД...")

print("\n--- Сценарий с ошибкой ---")
with DBTransaction(mock_db) as db:
    db["score"] = 200
    print("Пытаемся поделить на ноль...")
    x = 1 / 0  # Происходит ZeroDivisionError

print("Скрипт жив и продолжает работу!")

--- Успешный сценарий ---
>> Начинаем транзакцию...
Работаем с БД...
<< Всё отлично, делаем COMMIT!

--- Сценарий с ошибкой ---
>> Начинаем транзакцию...
Пытаемся поделить на ноль...
<< Ошибка ZeroDivisionError: division by zero. Делаем ROLLBACK!
Скрипт жив и продолжает работу!


In [29]:
class DBTransaction:
    def __init__(self, db_name: str):
        self.db_name = db_name

    def __enter__(self):
        print(f"[{self.db_name}] Открываем соединение, начинаем транзакцию...")
        return self  # Этот объект попадет в переменную после 'as'

    def query(self, sql: str):
        print(f"Выполняем: {sql}")

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            # Если внутри блока with произошла ошибка
            print(f"[{self.db_name}] ОШИБКА: {exc_val}. Делаем ROLLBACK!")
            # Возвращаем False, чтобы разработчик увидел ошибку в логах
            return False 
        
        # Если ошибок не было
        print(f"[{self.db_name}] Транзакция успешна. Делаем COMMIT!")
        print(f"[{self.db_name}] Закрываем соединение.\n")
        return True # (Хотя здесь можно вернуть и None, ошибки-то нет)

# Сценарий 1: Успех
with DBTransaction("PostgreSQL") as db:
    db.query("UPDATE users SET balance = balance - 100 WHERE id = 1")
    db.query("UPDATE users SET balance = balance + 100 WHERE id = 2")
# Вывод:
# [PostgreSQL] Открываем соединение, начинаем транзакцию...
# Выполняем: UPDATE users ...
# Выполняем: UPDATE users ...
# [PostgreSQL] Транзакция успешна. Делаем COMMIT!
# [PostgreSQL] Закрываем соединение.

# Сценарий 2: Провал
try:
    with DBTransaction("MySQL") as db:
        db.query("INSERT INTO orders (id, item) VALUES (1, 'Ноутбук')")
        raise RuntimeError("Сетевой кабель выдернули!")
except RuntimeError as e:
    print(f"Поймали ошибку снаружи: {e}")

# Вывод:
# [MySQL] Открываем соединение, начинаем транзакцию...
# Выполняем: INSERT INTO orders ...
# [MySQL] ОШИБКА: Сетевой кабель выдернули!. Делаем ROLLBACK!
# Поймали ошибку снаружи: Сетевой кабель выдернули!

[PostgreSQL] Открываем соединение, начинаем транзакцию...
Выполняем: UPDATE users SET balance = balance - 100 WHERE id = 1
Выполняем: UPDATE users SET balance = balance + 100 WHERE id = 2
[PostgreSQL] Транзакция успешна. Делаем COMMIT!
[PostgreSQL] Закрываем соединение.

[MySQL] Открываем соединение, начинаем транзакцию...
Выполняем: INSERT INTO orders (id, item) VALUES (1, 'Ноутбук')
[MySQL] ОШИБКА: Сетевой кабель выдернули!. Делаем ROLLBACK!
Поймали ошибку снаружи: Сетевой кабель выдернули!


### Вызываемые объекты: __call__


In [32]:
class RequestCounter:
    """Счетчик запросов, который помнит, сколько раз его вызывали."""
    def __init__(self, limit: int):
        self.limit = limit
        self.count = 0

    def __call__(self, url: str):
        if self.count >= self.limit:
            raise PermissionError("Превышен лимит запросов!")
        
        self.count += 1
        print(f"[{self.count}/{self.limit}] Отправка запроса на {url}...")
        # Здесь могла быть логика requests.get(url)

# Создаем экземпляр (состояние инициализировано)
fetcher = RequestCounter(limit=2)

# Вызываем ОБЪЕКТ как обычную функцию
fetcher("https://api.github.com")  # Вывод: [1/2] Отправка запроса...
fetcher("https://habr.com")        # Вывод: [2/2] Отправка запроса...
# fetcher("https://google.com")    # Выбросит PermissionError

[1/2] Отправка запроса на https://api.github.com...
[2/2] Отправка запроса на https://habr.com...
